# Superior Ensemble Training for Paperspace
## Automated One-Click Training Pipeline

This notebook executes the complete training pipeline using the superior ensemble trainer system. Just run all cells to train models automatically.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Setup paths and working directory
print("🔧 Setting up environment...")
project_root = Path("/notebooks/bot") if Path("/notebooks/bot").exists() else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📁 Working directory: {project_root}")

# Pull latest changes from GitHub
print("🔄 Pulling latest changes from GitHub...")
try:
    # Try to pull from the configured remote
    result = subprocess.run(
        ["git", "pull", "--ff-only"], 
        capture_output=True, 
        text=True, 
        cwd=project_root
    )
    
    if result.returncode == 0:
        print("✅ Successfully pulled latest changes")
        if result.stdout.strip():
            print(f"   {result.stdout.strip()}")
    else:
        print("⚠️ Git pull failed, continuing with current code")
        if result.stderr:
            print(f"   Error: {result.stderr.strip()}")
        
except Exception as e:
    print(f"⚠️ Could not pull from git: {e}")
    print("   Continuing with current code...")

print(f"🐍 Python path: {sys.path[0]}")

In [ ]:
# Import and execute the superior training system
print("🚀 Starting Superior Ensemble Training...")

try:
    # Import and run the Paperspace training pipeline
    from paperspace_mlops.paperspace_superior_training import main as run_training
    
    print("✅ Superior training system imported successfully")
    
    # Execute training with default parameters (all models, all symbols)
    print("🎯 Launching automated training pipeline...")
    result = run_training()
    
    if result == 0:
        print("🎉 Training completed successfully!")
    else:
        print(f"⚠️ Training completed with warnings (exit code: {result})")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("📝 Falling back to direct execution...")
    
    # Fallback: run the training script directly
    import subprocess
    result = subprocess.run([
        sys.executable, 
        "paperspace_mlops/paperspace_superior_training.py"
    ], capture_output=True, text=True)
    
    print("STDOUT:", result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    print(f"Return code: {result.returncode}")
    
except Exception as e:
    print(f"❌ Training failed: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# Post-training validation and S3 export
print("📊 Running post-training validation...")

try:
    # Check for trained models
    models_dir = Path("models")
    if models_dir.exists():
        model_files = list(models_dir.rglob("*.pt")) + list(models_dir.rglob("*.pkl")) + list(models_dir.rglob("*.zip"))
        print(f"✅ Found {len(model_files)} model files:")
        for model_file in model_files[:10]:  # Show first 10
            print(f"  📁 {model_file}")
        if len(model_files) > 10:
            print(f"  ... and {len(model_files) - 10} more")
    else:
        print("⚠️ No models directory found")
    
    # Optional: Export to S3 if credentials are available
    if os.getenv("AWS_ACCESS_KEY_ID") and os.getenv("AWS_SECRET_ACCESS_KEY"):
        print("☁️ AWS credentials found - exporting to S3...")
        try:
            import subprocess
            result = subprocess.run([
                sys.executable, 
                "paperspace_mlops/export_to_s3.py"
            ], capture_output=True, text=True)
            
            if result.returncode == 0:
                print("✅ Models exported to S3 successfully!")
            else:
                print(f"⚠️ S3 export failed: {result.stderr}")
        except Exception as e:
            print(f"⚠️ S3 export error: {e}")
    else:
        print("ℹ️ No AWS credentials - skipping S3 export")
    
    print("🎯 Training pipeline completed!")
    
except Exception as e:
    print(f"⚠️ Post-training validation error: {e}")
    # Don't raise - this is just validation